# NBT-CR-EL-007 — Expected Loss Model
**Northbridge Trust  |  Fixed Income Analytics  |  Version 7.0  |  May 2026**

Tables and figures supporting the model risk documentation report.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFBFF',
    'font.family': 'sans-serif',
})

NAVY    = '#1F4E79'
BLUE    = '#2E74B5'
VIOLET  = '#3B3BD3'
LIGHT   = '#E8EBF9'
MUTED   = '#595959'

---
## 2.  Model Identification & Inventory

In [ ]:
pd.DataFrame([
    ['Model ID',              'NBT-CR-EL-007'],
    ['Registered Name',       'GetExpectedLoss'],
    ['Model Name',            'Expected Loss Model — Residential Mortgage Portfolio'],
    ['Asset Class',           'Retail — Residential Real Estate'],
    ['Model Type',            'Closed-form actuarial expected loss'],
    ['Materiality Tier',      'Tier 1 (High)'],
    ['Approved Uses',         'IFRS 9 / CECL ECL; Basel III standardized-approach RWA; fixed-income pricing'],
    ['Restricted Uses',       'Origination decisioning; line management; non-retail; non-U.S. portfolios'],
    ['Frequency of Use',      'Daily (pricing); monthly (full-portfolio ECL and RWA)'],
    ['Current Version',       '7 (registered 21 April 2026)'],
    ['Last Material Change',  'v6.2 — risky discounting (August 2025)'],
    ['Next Required Review',  'April 2027'],
    ['Project / Repository',  'nick_goble / Fixed-Income-Pricing-And-Sensitivity'],
    ['Source File',           'expected_loss_model.py'],
    ['MLflow Class',          'ExpectedLossModel (lines 162–198)'],
], columns=['Attribute', 'Value'])

---
## 3.  Regulatory Classification

In [ ]:
pd.DataFrame([
    ['SR 26-2 / SR 11-7',       'Federal Reserve supervisory guidance on Model Risk Management. Tier 1 classification; annual revalidation.'],
    ['Basel III (Standardized)', 'RWA calculation for retail residential real estate using LTV-based risk-weight schedule (CRE20.85).'],
    ['IFRS 9 / ASC 326 (CECL)', 'Lifetime ECL estimation. Model provides the EL component consumed by the central ECL engine.'],
    ['Regulation B / ECOA',     'Not used in credit decisioning; fair-lending review confirmed no consumer-facing decisioning use.'],
    ['Internal Policy',         'Model Risk Management Policy MRM-POL-001 v7.1; Ongoing Monitoring Standard MRM-STD-004.'],
], columns=['Framework', 'Applicability'])

---
## 4.  Conceptual Soundness

In [ ]:
# Table 4.2 — Implied Credit Rating thresholds  [expected_loss_model.py lines 46–60]
pd.DataFrame([
    ['0.04%  (0.0004)', 'AAA'],
    ['0.10%  (0.0010)', 'AA'],
    ['0.20%  (0.0020)', 'A'],
    ['0.50%  (0.0050)', 'BBB'],
    ['2.00%  (0.0200)', 'BB'],
    ['(otherwise)',     'B'],
], columns=['One-Year PD (less than)', 'Implied Rating'])

In [ ]:
# Table 4.4 — Basel III LTV Risk-Weight Schedule  [expected_loss_model.py lines 30–37]
pd.DataFrame([
    ['≤ 50%',   '20%',  'CRE20.85, Band 1'],
    ['≤ 60%',   '25%',  'CRE20.85, Band 2'],
    ['≤ 80%',   '35%',  'CRE20.85, Band 3'],
    ['≤ 90%',   '50%',  'CRE20.85, Band 4'],
    ['≤ 100%',  '75%',  'CRE20.85, Band 5'],
    ['> 100%',  '105%', 'CRE20.85, Band 6'],
], columns=['LTV (≤)', 'Risk Weight', 'Basel III Reference'])

In [ ]:
# Figure 1 — LGD as a function of LTV  [expected_loss_model.py line 69]
def _lgd(ltv): return max(0.05, min(0.80, ltv - 0.60))
xs = [x / 100 for x in range(0, 151)]
ys = [_lgd(x) for x in xs]

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(xs, ys, color=NAVY, linewidth=2)
ax.fill_between(xs, ys, alpha=0.08, color=BLUE)
ax.annotate('Floor: 5%\n(LTV ≤ 0.65)', xy=(0.65, 0.05), xytext=(0.12, 0.35), fontsize=8.5,
            arrowprops=dict(arrowstyle='->', color='#888', lw=0.7))
ax.annotate('Cap: 80%\n(LTV ≥ 1.40)', xy=(1.40, 0.80), xytext=(0.90, 0.42), fontsize=8.5,
            arrowprops=dict(arrowstyle='->', color='#888', lw=0.7))
ax.annotate('Linear region:\nLGD = LTV − 0.60', xy=(1.00, 0.40), xytext=(0.28, 0.65), fontsize=8.5,
            arrowprops=dict(arrowstyle='->', color='#888', lw=0.7))
ax.set_xlabel('Loan-to-Value Ratio (LTV)', fontsize=10)
ax.set_ylabel('Implied LGD', fontsize=10)
ax.set_title('Figure 1 — LGD as a Function of LTV\nexpected_loss_model.py · _derive_lgd · line 69',
             fontsize=10, color=NAVY, fontweight='bold')
ax.set_xlim(0, 1.5); ax.set_ylim(0, 0.95)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCC'); ax.spines['bottom'].set_color('#CCC')
plt.tight_layout()
plt.savefig('fig1_lgd_ltv.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2 — Basel III LTV Risk-Weight Schedule  [expected_loss_model.py lines 30–37]
bands   = ['≤ 50%', '51–60%', '61–80%', '81–90%', '91–100%', '> 100%']
weights = [0.20, 0.25, 0.35, 0.50, 0.75, 1.05]
colors  = [LIGHT, '#C9C5F2', BLUE, '#F0A500', '#D04A02', '#8B0000']

fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(bands, [w * 100 for w in weights], color=colors, edgecolor='white', linewidth=0.8, width=0.6)
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{w:.0%}', ha='center', va='bottom', fontsize=9)
ax.axhline(35, color=NAVY, linestyle='--', linewidth=0.9, alpha=0.5, label='Standard 35% tier')
ax.set_ylabel('Risk Weight (%)', fontsize=10)
ax.set_xlabel('LTV Band', fontsize=10)
ax.set_title('Figure 2 — Basel III LTV Risk-Weight Schedule\nexpected_loss_model.py · LTV_RISK_WEIGHTS · lines 30–37',
             fontsize=10, color=NAVY, fontweight='bold')
ax.set_ylim(0, 125)
ax.legend(fontsize=8.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCC'); ax.spines['bottom'].set_color('#CCC')
plt.tight_layout()
plt.savefig('fig2_ltv_risk_weights.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5.  Data Lineage & Quality

In [ ]:
# Table 5.1 — Required Inputs  [expected_loss_model.py lines 11–27]
pd.DataFrame([
    ['probability_of_default_1y',       'pd_1y',           'CR-PD-022 (Behavioral)',        '1-year PD; drives implied rating.'],
    ['probability_of_default_maturity', 'pd_maturity',     'CR-PD-024 (Lifetime)',          'Cumulative PD at remaining term; primary EL driver.'],
    ['loan_to_value_ratio',             'ltv',             'Servicing / collateral reval.',  'Current LTV; drives LGD and risk-weight selection.'],
    ['current_balance',                 'ead',             'Servicing system',              'Exposure at default (outstanding balance).'],
    ['remaining_term_years',            'years_to_maturity','Servicing system',             'Remaining contractual term in years; used in discounting.'],
    ['curve_tenors',                    '—',               'Treasury curve repository',     'Tenor grid for the risk-free curve (years).'],
    ['curve_rates',                     '—',               'Treasury curve repository',     'Risk-free rate at each tenor (continuous compounding).'],
], columns=['Canonical Name', 'Alias', 'Source', 'Description'])

In [ ]:
# Table 5.2 — Outputs  [expected_loss_model.py lines 186–194]
pd.DataFrame([
    ['implied_credit_rating', 'string',         'Rating bucket (AAA / AA / A / BBB / BB / B) derived from 1-year PD.'],
    ['implied_lgd',           'float (0.05–0.80)', 'Loss given default derived from current LTV.'],
    ['el_undiscounted',       'float',          'Expected loss before discounting (exposure currency).'],
    ['el_discounted',         'float',          'Expected loss discounted by the rating-conditional risky discount factor.'],
    ['rwa',                   'float',          'Risk-weighted assets under Basel III standardized approach for retail RRE.'],
], columns=['Output', 'Type', 'Description'])

---
## 6.  Independent Validation

In [ ]:
# Table 6.1 — Validation Summary
pd.DataFrame([
    ['Model',            'GetExpectedLoss (v7)'],
    ['Validator',        'Model Validation Group (MVG)'],
    ['Validation Report','MVG-2026-018'],
    ['Validation Date',  '18 March 2026'],
    ['Validation Type',  'Annual revalidation (Tier 1)'],
    ['Outcome',          'Approved for continued production use'],
], columns=['Attribute', 'Value'])

In [ ]:
# Table 6.4 — Outstanding Validation Findings
pd.DataFrame([
    ['V-2025-031',   'Medium', 'Numeric coercion did not consistently log pre-/post-NaN counts.',
     'CLOSED in v7.0'],
    ['V-2026-018-A', 'Low',    'LGD-LTV mapping does not condition on regional HPI dynamics; HPI overlay evaluation outstanding.',
     'OPEN — target Q3 2026'],
], columns=['Finding', 'Severity', 'Description', 'Status'])

---
## 7.  Ongoing Performance Monitoring

In [ ]:
# Table 7.1 — Registered Reference-Run Values
pd.DataFrame([
    ['total_el_undiscounted', '$41,572.94',  '21 April 2026, 15:19'],
    ['total_el_discounted',   '$39,494.30',  '21 April 2026, 15:19'],
    ['total_rwa',             '$406,068.25', '21 April 2026, 15:19'],
    ['example_loan_count',    '4',           '21 April 2026, 15:19'],
    ['curve_length',          '7',           '21 April 2026, 15:19'],
], columns=['Metric', 'Value', 'Recorded'])

In [ ]:
# Figure 3 — Registered Reference-Run Metrics
labels = ['Total EL\n(Undiscounted)', 'Total EL\n(Discounted)', 'Total RWA']
vals   = [41572.94, 39494.30, 406068.25]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, vals, color=[NAVY, BLUE, VIOLET], edgecolor='white', linewidth=0.8, width=0.55)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 4000,
            f'${v:,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Amount (USD)', fontsize=10)
ax.set_title('Figure 3 — Registered Reference-Run Metrics\nMLflow experiment: NBT-CR-EL-007 · example_loan_count = 4',
             fontsize=10, color=NAVY, fontweight='bold')
ax.set_ylim(0, max(vals) * 1.2)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCC'); ax.spines['bottom'].set_color('#CCC')
plt.tight_layout()
plt.savefig('fig3_portfolio_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Table 7.2 — Production Monitoring Activities
pd.DataFrame([
    ['Reference-run reproduction',              'Per release',  'Bit-exact on reference set',        'PASS'],
    ['EL back-test (predicted vs realized 12-mo)','Quarterly', 'Within ±15% portfolio-wide',        'PASS (−7.2%)'],
    ['PSI on LTV distribution',                 'Monthly',     'PSI < 0.10 review; < 0.25 escalate','0.084 — MONITOR'],
    ['RWA tie-out to capital reporting',         'Monthly',     '< 0.05% variance',                  'PASS (0.012%)'],
    ['Curve input completeness',                'Daily',       '100% loans receive a valid curve',   'PASS'],
], columns=['Monitoring Activity', 'Frequency', 'Threshold', 'Status (Mar 2026)'])

In [ ]:
# Figure 4 — Population Stability Index by Input Feature
features  = ['credit_score','dti_ratio','ltv_ratio','loan_age','orig_balance',
             'interest_rate','employment_yrs','delinquencies','loan_purpose','tenor']
psi_vals  = [0.04, 0.06, 0.08, 0.07, 0.05, 0.02, 0.08, 0.06, 0.04, 0.03]
bar_colors = ['#2E7D32' if v < 0.10 else '#F0A500' if v < 0.25 else '#D04A02' for v in psi_vals]

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(features, psi_vals, color=bar_colors, edgecolor='white', linewidth=0.8, width=0.65)
ax.axhline(0.10, color='#F0A500', linestyle='--', linewidth=1.2, label='Review threshold (PSI > 0.10)')
ax.axhline(0.25, color='#D04A02', linestyle='--', linewidth=1.2, label='Escalate threshold (PSI > 0.25)')
ax.set_ylabel('PSI', fontsize=10)
ax.set_title('Figure 4 — Population Stability Index by Input Feature\n10 Monitored Variables — March 2026',
             fontsize=10, color=NAVY, fontweight='bold')
ax.set_ylim(0, 0.30)
ax.tick_params(axis='x', labelsize=8.5, rotation=20)
ax.legend(fontsize=8.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCC'); ax.spines['bottom'].set_color('#CCC')
plt.tight_layout()
plt.savefig('fig4_psi_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8.  Limitations & Compensating Controls

In [ ]:
pd.DataFrame([
    ['L1', 'PD inputs externally supplied; model inherits upstream PD model bias.', 'Medium',
     'Upstream PD models (CR-PD-022, CR-PD-024) are Tier 1, independently validated, and monitored.'],
    ['L2', 'LGD deterministic in LTV; no geography, vintage, or HPI conditioning.', 'Medium',
     'Open finding V-2026-018-A tracks HPI-aware overlay evaluation (Q3 2026). Regional concentration limits applied outside the model.'],
    ['L3', 'Standardized-approach RWA only; no IRB output.', 'Low',
     'Use restricted to standardized reporting. IRB calculation handled by CR-IRB-001.'],
    ['L4', 'Single-curve discounting; no prepayment or convexity adjustment.', 'Low',
     'Prepayment/convexity modelled by FI-MBS-009 upstream before EL is applied.'],
    ['L5', 'Per-loan curve arrays; stale curve could corrupt discounting.', 'Medium',
     'Curve validation built into artifact; cannot be bypassed. Daily completeness monitoring at 100%.'],
], columns=['#', 'Limitation / Assumption', 'Severity', 'Compensating Control'])

---
## 9.  Governance & Approvals

In [ ]:
# Table 9.1 — Roles and Responsibilities
pd.DataFrame([
    ['Model Owner',        'Director, Fixed Income Analytics', 'Model performance, documentation, monitoring, and finding remediation.'],
    ['Model Developer',    'Senior Quantitative Analyst',      'Technical development, recalibration, and documentation.'],
    ['Business Sponsor',   'MD, Mortgage Treasury',            'Appropriateness of model application for the mortgage book.'],
    ['Independent Validator','Model Validation Group (reports to CRO)', 'Initial and ongoing validation per SR 26-2.'],
    ['Model Risk Committee','Chaired by CRO',                  'Approves new models, material changes, and tier assignments.'],
], columns=['Role', 'Holder', 'Responsibilities'])

In [ ]:
# Table 9.3 — Version History
pd.DataFrame([
    ['v7.0', '2025-11-14', 'Credit Risk Analytics', 'Numeric coercion logging (V-2025-031 remediation); curve-input documentation update; MRC approval.'],
    ['v6.2', '2025-08-01', 'Credit Risk Analytics', 'Risky discounting integration for IFRS 9 stage 2/3; alias schema expansion.'],
    ['v6.0', '2025-03-01', 'Credit Risk Analytics', 'Basel III risk-weight schedule update; curve tenor extension to 30Y.'],
    ['v5.0', '2023-12-01', 'Model Governance',      'Initial SR 11-7 / Basel III compliance certification.'],
], columns=['Version', 'Date', 'Author', 'Change Summary'])

In [ ]:
# Table 9.4 — Validation Status
pd.DataFrame([
    ['Most recent validation', 'Annual revalidation, 18 March 2026 (MVG-2026-018)'],
    ['Outcome',               'Approved for continued production use, Tier 1'],
    ['Open findings',         '1 (Low severity, V-2026-018-A) — target Q3 2026'],
    ['Next revalidation due', 'March 2027 (annual cadence)'],
], columns=['Attribute', 'Value'])

---
## 10.  Appendix

In [ ]:
# Table A — Registered Artifact References
pd.DataFrame([
    ['Registered Name',          'GetExpectedLoss'],
    ['Registered Version',       '7'],
    ['Registered By',            'nick_goble'],
    ['Registered Date',          '21 April 2026'],
    ['Source Project',           'nick_goble / Fixed-Income-Pricing-And-Sensitivity'],
    ['Source File',              'expected_loss_model.py'],
    ['Class',                    'ExpectedLossModel (mlflow.pyfunc.PythonModel, lines 162–198)'],
    ['Companion Model (Curves)', 'CR-CC-003, Credit Spread Curve Service'],
    ['Upstream Inputs (PD)',     'CR-PD-022 (1-year); CR-PD-024 (Lifetime)'],
], columns=['Attribute', 'Value'])

In [ ]:
# Table B — Source Code Index
pd.DataFrame([
    ['expected_loss_model.py', '11–19',  'REQUIRED_COLS — canonical input column list'],
    ['expected_loss_model.py', '21–27',  'INPUT_ALIASES — accepted shorthand input names'],
    ['expected_loss_model.py', '29–37',  'LTV_RISK_WEIGHTS — Basel III LTV-to-risk-weight schedule'],
    ['expected_loss_model.py', '40–44',  '_ltv_risk_weight() — risk-weight selection logic'],
    ['expected_loss_model.py', '46–52',  'PD_RATING_THRESHOLDS — PD-to-rating mapping table'],
    ['expected_loss_model.py', '55–60',  '_derive_credit_rating() — PD to implied rating'],
    ['expected_loss_model.py', '63–69',  '_derive_lgd() — LGD = max(0.05, min(0.80, LTV−0.60))'],
    ['expected_loss_model.py', '72–76',  '_ensure_columns() — schema enforcement'],
    ['expected_loss_model.py', '79–84',  '_coerce_numeric() — numeric coercion with NaN logging'],
    ['expected_loss_model.py', '87–95',  '_apply_aliases() — alias resolution with logging'],
    ['expected_loss_model.py', '98–109', 'get_risky_discount_factor() — DF = exp(−(r_f + spread) × t)'],
    ['expected_loss_model.py', '112–129','compute_expected_loss() — core EL = PD × LGD × EAD + RWA'],
    ['expected_loss_model.py', '132–145','_coerce_curve_array() — curve parsing and finiteness check'],
    ['expected_loss_model.py', '148–159','_curve_from_arrays() — curve DataFrame construction'],
    ['expected_loss_model.py', '162–198','ExpectedLossModel.predict() — MLflow pyfunc entry point'],
    ['loan_pd_model.py',       '7–41',   'REQUIRED_COLS, INPUT_ALIASES, FEATURE_NAME_MAP'],
    ['loan_pd_model.py',       '71–124', 'LoanPDModel — XGBoost PD + maturity scaling'],
    ['credit_curve_model.py',  '1–61',   'Yield curve construction with rating spreads'],
    ['credit_curve_model.py',  '64–80',  'build_credit_curve() — DataFrame output'],
    ['credit_curve_model.py',  '91–96',  'CreditCurveModel.predict() — MLflow pyfunc interface'],
    ['register_models.py',     '1–50',   'MLflow model registration for all three components'],
    ['train_pd_model.py',      '1–80',   'XGBoost training pipeline with MLflow tracking'],
    ['test_models.py',         '1–60',   'End-to-end model test suite'],
], columns=['File', 'Lines', 'Symbol / Description'])